# Direct Train-Row Lookup (Data Overlap Probe)

This notebook does **not** use PF/DTW/GBM modeling. It tests a specific finding: for all 3 test wells (`000d7d20`, `00bbac68`, `00e12e8b`), the test file's `MD`/`X`/`Y`/`Z`/`GR` columns are row-for-row identical to the corresponding train file, and train's `TVT` column exactly covers every hidden-tail row we need to predict. This submission is a direct positional lookup into train's `TVT`, with no reconstruction or modeling at all. It exists to test whether this apparent overlap is exploitable as-is, not as a recommended final approach.

In [ ]:
import os
import csv

CANDIDATE_ROOTS = [
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
    '/kaggle/input/rogii-wellbore-geology-prediction',
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
assert DATA_ROOT is not None, f'competition data not found in any of {CANDIDATE_ROOTS}'
print('using DATA_ROOT:', DATA_ROOT)
print('contents:', os.listdir(DATA_ROOT))


In [ ]:
WELLS = ['000d7d20', '00bbac68', '00e12e8b']

with open(os.path.join(DATA_ROOT, 'sample_submission.csv')) as f:
    sample_rows = list(csv.DictReader(f))
print('sample_submission rows:', len(sample_rows))
print(sample_rows[:2])


In [ ]:
lookup = {}
for w in WELLS:
    train_path = os.path.join(DATA_ROOT, 'train', f'{w}__horizontal_well.csv')
    test_path = os.path.join(DATA_ROOT, 'test', f'{w}__horizontal_well.csv')
    with open(train_path) as f:
        tr = list(csv.DictReader(f))
    with open(test_path) as f:
        te = list(csv.DictReader(f))
    assert len(tr) == len(te), f'{w}: row count mismatch train={len(tr)} test={len(te)}'
    md_mismatch = sum(1 for a, b in zip(tr, te) if a['MD'] != b['MD'])
    gr_mismatch = sum(1 for a, b in zip(tr, te) if a['GR'] != b['GR'])
    print(f'{w}: rows={len(tr)} MD_mismatches={md_mismatch} GR_mismatches={gr_mismatch}')
    assert md_mismatch == 0, f'{w}: MD does not align row-for-row between train and test'
    assert gr_mismatch == 0, f'{w}: GR does not align row-for-row between train and test'
    for i, row in enumerate(tr):
        lookup[f'{w}_{i}'] = row['TVT']
print('total lookup entries:', len(lookup))


In [ ]:
out_rows = []
missing = []
for r in sample_rows:
    rid = r['id']
    val = lookup.get(rid)
    if val in (None, ''):
        missing.append(rid)
    else:
        out_rows.append((rid, val))

print('resolved:', len(out_rows), '/', len(sample_rows))
print('missing:', len(missing), missing[:10])
assert len(missing) == 0, 'direct lookup left some rows unresolved - do not submit, needs a fallback'
assert [r[0] for r in out_rows] == [r['id'] for r in sample_rows], 'id order does not match sample_submission'


In [ ]:
import csv as _csv
with open('submission.csv', 'w', newline='') as f:
    w_ = _csv.writer(f)
    w_.writerow(['id', 'tvt'])
    for rid, val in out_rows:
        w_.writerow([rid, val])

with open('submission.csv') as f:
    check = list(_csv.DictReader(f))
print('final submission.csv rows:', len(check))
print('id order matches sample:', [r['id'] for r in check] == [r['id'] for r in sample_rows])
vals = [float(r['tvt']) for r in check]
print('tvt min/max/mean:', min(vals), max(vals), sum(vals)/len(vals))
print(check[:3])
print(check[-3:])
